In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import random
import scipy.sparse
from typing import Dict, List, Union, Optional

In [ ]:
xenium = sc.read_h5ad('data/gut_data/gut_hs_XeniumAdultColonNicheCompass_AM_21102024_150114_raw.h5ad')

In [ ]:
adata_sc = sc.read_h5ad('data/gut_data/gut_hs_full_annotated_AM_06032025_140458_raw.h5ad')

In [ ]:
adata_sc = adata_sc[adata_sc.obs['age_group'] == 'adult']

In [ ]:
cell_type_mapping = {
    # Epithelial cells
    "BEST4+ epithelial": "BEST4+ epithelial",
    "Enterocyte": "Enterocyte",
    "Stem cells": "Stem cells",
    "TA": "TA",
    "Colonocyte": "Colonocyte",
    "Goblet cells": ["Goblet cell", "mature Goblet cell", "immature Goblet cell"],
    "EECs": ["EECs", "Deep crypt secretory cells", "M/X cells (MLN/GHRL+)", "D cells (SST+)", "I cells (CCK+)"],
    "Tuft cells": "Tuft",
    "Microfold cell": "Microfold cells",
    
    # Stromal cells
    "Myofibroblasts": "Myofibroblast (RSPO2+)",
    "Fibroblasts": ["Stromal 1 (ADAMDEC1+)", "Transitional Stromal 3 (C3+)", "Stromal 3 (KCNN3+)", "Stromal 2 (NPY+)", "T reticular", "Lymphoid_Stroma"],
    "Pericytes": ["Pericyte", "Immature pericyte", "Contractile pericyte", "Angiogenic pericyte"],
    "Mesothelium": "Mesothelium",
    
    # Immune cells
    "B cells": ["Memory B cells", "Naive B cells", "Cycling B cell", "Germinal Center B cells"],
    "Plasma cells": ["Plasma cells", "IgA plasma cell", "IgM plasma cell", "Cycling plasma cell"],
    "Macrophages": "Macrophages",
    "Monocytes": "Macrophages",
    "CD8 T": ["CD8 Tmem", "Naive CD8 T cells", "Activated CD8 T", "TRM cytotoxic T"],
    "CD4 T": ["Activated CD4 T", "Naive CD4 T cells", "Activated T", "Tfh"],
    "Mast cells": "Mast cells",
    "Tregs": "Treg",
    "NK": ["NK cells", "NK T cell"],
    "DC": "cDC2",
    "ILCs": "ILC3",
    "gdT": "gamma delta T cells",
    "Immune Cycling cells": ["Cycling T", "Cycling B cell", "Cycling plasma cell", "Cycling stromal"],
    
    # Endothelial cells
    "Endothelial cells": ["Arterial EC", "Venous EC", "LEC", "arterial capillary"],
    "Mature arterial EC": "Arterial EC",
    "Mature venous EC": "Venous EC",
    "LEC": "LEC",
    "Lymphatics": "LEC",
    "arterial capillary": "arterial capillary",
    
    # Other
    "Glial cells": ["Mesoderm 1(HAND1+)", "Mesoderm 2(ZEB2+)"],
    "Adult Glia": ["Mesoderm 1(HAND1+)", "Mesoderm 2(ZEB2+)"]
}

In [ ]:
def create_synthetic_spatial_dataset(
    xenium_adata: ad.AnnData,
    scrna_adata: ad.AnnData,
    cell_type_mapping: Dict[str, Union[str, List[str]]],
    cell_type_col_xenium: str = "cell_type",
    cell_type_col_scrna: str = "cell_type",
    spatial_cols: list = ["x", "y"],
    random_seed: Optional[int] = 42
) -> ad.AnnData:
    """
    Create a synthetic spatial dataset by combining cell metadata and spatial coordinates
    from a Xenium dataset with gene expression counts from a scRNA-seq dataset.
    
    Parameters
    ----------
    xenium_adata : AnnData
        Xenium spatial dataset with cell metadata and spatial coordinates
    scrna_adata : AnnData
        scRNA-seq dataset with gene expression counts
    cell_type_mapping : Dict[str, Union[str, List[str]]]
        Dictionary mapping cell types from Xenium to cell types in scRNA-seq.
        Values can be either a single string or a list of strings.
    cell_type_col_xenium : str, optional
        Column name in xenium_adata.obs containing cell type information, by default "cell_type"
    cell_type_col_scrna : str, optional
        Column name in scrna_adata.obs containing cell type information, by default "cell_type"
    spatial_cols : list, optional
        Column names in xenium_adata.obsm['spatial'] for spatial coordinates, by default ["x", "y"]
    random_seed : int, optional
        Random seed for reproducibility, by default 42
        
    Returns
    -------
    AnnData
        Synthetic spatial dataset with gene expression counts from scRNA-seq
        and spatial coordinates from Xenium
    """
    # Set random seed for reproducibility
    if random_seed is not None:
        np.random.seed(random_seed)
        random.seed(random_seed)
    
    # Check if spatial columns exist in xenium_adata
    if 'spatial' not in xenium_adata.obsm:
        raise ValueError("Xenium dataset does not have 'spatial' in obsm")
    
    # Create a new AnnData object with the same shape as the scRNA-seq dataset
    # but with cells from the Xenium dataset
    synthetic_adata = ad.AnnData(
        X=np.zeros((xenium_adata.n_obs, scrna_adata.n_vars), dtype=np.float32),
        var=scrna_adata.var.copy(),
        obs=xenium_adata.obs.copy()
    )
    
    # Copy spatial coordinates
    synthetic_adata.obsm['spatial'] = xenium_adata.obsm['spatial'].copy()
    
    # Print some debugging information
    print(f"Xenium dataset: {xenium_adata.shape[0]} cells")
    print(f"scRNA-seq dataset: {scrna_adata.shape[0]} cells")
    print(f"Cell type column in Xenium: {cell_type_col_xenium}")
    print(f"Cell type column in scRNA-seq: {cell_type_col_scrna}")
    
    # Pre-compute the indices of cells for each cell type in scRNA-seq data
    scrna_cell_type_indices = {}
    for cell_type in scrna_adata.obs[cell_type_col_scrna].unique():
        mask = scrna_adata.obs[cell_type_col_scrna] == cell_type
        scrna_cell_type_indices[cell_type] = np.where(mask.values)[0]
    
    # For each cell in the Xenium dataset, find a matching cell in the scRNA-seq dataset
    cell_matches = 0
    cell_mismatches = 0
    
    for i in range(xenium_adata.n_obs):
        # Get the cell type from Xenium
        xenium_cell_type = xenium_adata.obs[cell_type_col_xenium].iloc[i]
        
        # Map to the corresponding scRNA-seq cell type(s)
        if xenium_cell_type in cell_type_mapping:
            scrna_cell_types = cell_type_mapping[xenium_cell_type]
            
            # Convert to list if it's a single string
            if isinstance(scrna_cell_types, str):
                scrna_cell_types = [scrna_cell_types]
            
            # Collect all possible cell indices across all matching cell types
            all_matching_indices = []
            for scrna_cell_type in scrna_cell_types:
                if scrna_cell_type in scrna_cell_type_indices:
                    all_matching_indices.extend(scrna_cell_type_indices[scrna_cell_type])
            
            if all_matching_indices:
                # Randomly select one matching cell index
                random_idx = random.choice(all_matching_indices)
                
                # Copy gene expression counts from the selected scRNA-seq cell
                if scipy.sparse.issparse(scrna_adata.X):
                    synthetic_adata.X[i] = scrna_adata.X[random_idx].toarray().flatten()
                else:
                    synthetic_adata.X[i] = scrna_adata.X[random_idx]
                
                cell_matches += 1
            else:
                cell_mismatches += 1
                if i < 5:  # Print just a few examples to avoid flooding the output
                    print(f"No matching cells found for {xenium_cell_type} -> {scrna_cell_types}")
        else:
            cell_mismatches += 1
            if i < 5:  # Print just a few examples to avoid flooding the output
                print(f"No mapping found for cell type: {xenium_cell_type}")
    
    print(f"Successfully matched {cell_matches} cells")
    print(f"Failed to match {cell_mismatches} cells")
    
    # If too many mismatches, print an example of the mapping to help debug
    if cell_mismatches > 0:
        print("\nExample of mapping dictionary:")
        for k, v in list(cell_type_mapping.items())[:5]:  # Show first 5 entries
            print(f"  {k} -> {v}")
    
    return synthetic_adata

In [ ]:
synthetic_adata = create_synthetic_spatial_dataset(
    xenium_adata=xenium, scrna_adata=adata_sc,
    cell_type_mapping=cell_type_mapping, cell_type_col_xenium="C_scANVI",  cell_type_col_scrna="cell_states",  
    spatial_cols=["x", "y"] 
)

Xenium dataset: 274037 cells
scRNA-seq dataset: 10767 cells
Cell type column in Xenium: C_scANVI
Cell type column in scRNA-seq: cell_states
Successfully matched 274037 cells
Failed to match 0 cells


In [ ]:
synthetic_adata.write_h5ad('data/gut_data/synthetic_spatial_43K_genes.h5ad')